# Lesson 03 - Agentic Design Patterns

In this lesson, we explore three foundational design patterns for building effective AI agents:

1. **Clear Agent Instructions** — Crafting precise, role-defining prompts that guide agent behavior
2. **Structured Output with Pydantic Models** — Ensuring agents return predictable, validated data
3. **Single Responsibility Agents** — Designing focused agents that each do one thing well

We'll apply each pattern to a **travel destination recommender** scenario, progressively building a system that can suggest destinations, check availability, and handle logistics.

## Setup

In [1]:
%pip install agent-framework azure-ai-projects azure-identity pydantic --quiet

Note: you may need to restart the kernel to use updated packages.


In [6]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
import openai as openai_sdk
from typing import Annotated
from pydantic import BaseModel, Field
from azure.identity import DefaultAzureCredential
from azure.ai.projects import AIProjectClient
from azure.ai.projects.models import PromptAgentDefinition
from azure.ai.projects.models import PromptAgentDefinition, Tool, FunctionTool
from openai.types.responses.response_input_param import FunctionCallOutput

## Pattern 1: Clear Agent Instructions

The most impactful pattern is also the simplest: writing clear, detailed instructions for your agent.

Good instructions define:
- **Who** the agent is (persona and tone)
- **What** it should do (step-by-step responsibilities)
- **How** it should behave (constraints and style)

Below, we create a travel concierge agent with explicit instructions that shape every response it produces.

In [3]:
project_client = AIProjectClient(
    endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    credential=DefaultAzureCredential()
)

agent = project_client.agents.create_version(
    agent_name="TravelConcierge",
    definition=PromptAgentDefinition(
        model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
        instructions="""You are a luxury travel concierge named Alex. Your role is to:
            1. Understand the traveler's preferences (budget, climate, activities)
            2. Check destination availability before making recommendations
            3. Provide detailed, personalized travel suggestions
            4. Always mention visa requirements and best travel seasons
            Be warm, professional, and enthusiastic about travel."""
    )
)

openai = project_client.get_openai_client()

response = openai.responses.create(
    input="I'd love a week-long vacation somewhere with great food and history. Budget around $2500.",
    extra_body={
        "agent_reference": {
            "name": agent.name,
            "type": "agent_reference",
        }
    },
)

print(response.output_text)

That's wonderful! Let's find the perfect destination for your culinary and historical adventures. Could you let me know your preferred travel dates or season, and whether you have any climate preferences? This will help me tailor the recommendations just for you.


## Pattern 2: Structured Output with Pydantic Models

Free-form text is useful for conversation, but downstream systems need structured data.
By pairing **Pydantic models** with a **tool function**, we can:

- Define an exact schema for the agent's output
- Validate responses automatically
- Integrate agent results into application logic reliably

We also introduce a tool that returns destination details so the agent grounds its recommendations in real data.

In [17]:
class DestinationRecommendation(BaseModel):
    destination: str
    available: bool
    best_season: str
    highlights: list[str]
    estimated_budget_usd: int


class TravelRecommendations(BaseModel):
    recommendations: list[DestinationRecommendation]
    personalized_note: str

class DestinationArgs(BaseModel):
    destination: str = Field(description="The destination to look up")

def get_destination_details(destination: Annotated[str, "The destination to look up"]) -> str:
    """Get details about a vacation destination."""
    details = {
        "Barcelona": "Available. Best: May-Jun. Beach, architecture, nightlife. ~$2000/week",
        "Tokyo": "Available. Best: Mar-Apr. Culture, food, technology. ~$2500/week",
        "Cape Town": "Not available. Best: Nov-Mar. Nature, wine, adventure. ~$1800/week",
        "Bali": "Available. Best:June-July, beach. ~2000/week"
    }
    return details.get(destination, f"{destination}: No information available.")

destination_tool = FunctionTool(
    name="get_destination_details",
    description="Get details about a vacation destination.",
    parameters= {
       "type": "object",
        "properties": {
          "destination": { "type": "string" }
        },
        "required": ["destination"] 
    }
)



openai = project_client.get_openai_client()

tools = [
   openai_sdk.pydantic_function_tool(
        DestinationArgs,
        name="get_destination_details",
        description="Get details about a vacation destination.",
    )

]
messages = [
        {
            "role": "system",
            "content":"You are a travel expert. Recommend destinations based on traveler preferences. Use the get_destination_details tool."
        },
        {
            "role": "user",
            "content": "Recommend one destination for a beach travel budget $3000"
        }
    ]

completion = openai.beta.chat.completions.parse(
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    messages=messages,
    tools=tools,
    tool_choice={
        "type": "function",
        "function": {"name": "get_destination_details"}
    }
)

assistant_message = completion.choices[0].message
print("Message", assistant_message.parsed)
tool_call = (assistant_message.tool_calls or [])[0]
tool_args: str = tool_call.function.arguments

print("tool args:", tool_args)

tool_result = get_destination_details(tool_args)
print("tool result", tool_result)

messages.append(
    {
        "role": "assistant",
        "content": None,
        "tool_calls": [
            {
                "id": tool_call.id,
                "type": "function",
                "function": {
                    "name": tool_call.function.name,
                    "arguments": tool_call.function.arguments
                }
            }
        ]
    }
)

messages.append(
    {
        "role": "tool",
        "tool_call_id": tool_call.id,
        "content": json.dumps(tool_result)
    }
)

final = openai.beta.chat.completions.parse(
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    messages=messages,
    tools=tools,
    tool_choice="none",
    response_format=TravelRecommendations
)

typed_response: TravelRecommendations = final.choices[0].message.parsed

print("Typed final object:", typed_response)
print("PersonalisedNote", typed_response.personalized_note)


Message None
tool args: {"destination":"Maldives"}
tool result {"destination":"Maldives"}: No information available.
Typed final object: recommendations=[DestinationRecommendation(destination='Maldives', available=False, best_season='November to April', highlights=['Overwater bungalows', 'Snorkeling and diving', 'White sandy beaches', 'Crystal clear waters'], estimated_budget_usd=3000)] personalized_note="The Maldives is an idyllic beach destination known for its luxurious overwater bungalows and incredible marine life. While information on the current situation or availability isn't available, it is generally a great choice for a budget of $3000, particularly during the best season from November to April when the weather is perfect for exploring the islands and enjoying water activities."
PersonalisedNote The Maldives is an idyllic beach destination known for its luxurious overwater bungalows and incredible marine life. While information on the current situation or availability isn't 

## Pattern 3: Single Responsibility Agents

Complex tasks benefit from splitting work across multiple focused agents, each with a single responsibility:

- A **Destination Expert** that knows about places and availability
- A **Logistics Planner** that handles flights, hotels, and itineraries

This mirrors the software engineering principle of *separation of concerns* — each agent is easier to test, maintain, and improve independently.

In [29]:
destination_messages = [
        {
            "role": "system",
            "content":"""You are a destination research specialist. Your only job is to:
1. Evaluate destinations based on traveler preferences
2. Check availability using the provided tool
3. Return a short ranked list with pros/cons
Do NOT discuss flights, hotels, or logistics — another agent handles that."""
        },
        {
            "role": "user",
            "content": "Recommend one destination for a beach travel budget $5000"
        },
        {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }
            ]
        },
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(tool_result)
        }
    ]




destination_response = openai.beta.chat.completions.parse(
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    messages=destination_messages,
    tools=tools,
    tool_choice="none",
    response_format=TravelRecommendations
)

typed_dest_response: TravelRecommendations = destination_response.choices[0].message.parsed

print("Typed destination object:", typed_dest_response)
print("Destination PersonalisedNote", typed_dest_response.personalized_note)

logistic_user_message =  f"Plan a week-long trip based on this recommendation:\n{typed_dest_response.recommendations[0].destination}"
print("Logistic user message:", logistic_user_message)

logistic_message = [
        {
            "role": "system",
            "content":"""You are a travel logistics planner. Your only job is to:
1. Create a day-by-day itinerary for the chosen destination
2. Suggest flight and hotel options within the stated budget
3. Note visa requirements and travel insurance recommendations
Do NOT recommend destinations — another agent handles that."""
        },
        {
            "role": "user",
            "content": logistic_user_message
        },
        {
            "role": "assistant",
            "content": None,
            "tool_calls": [
                {
                    "id": tool_call.id,
                    "type": "function",
                    "function": {
                        "name": tool_call.function.name,
                        "arguments": tool_call.function.arguments
                    }
                }
            ]
        },
        {
            "role": "tool",
            "tool_call_id": tool_call.id,
            "content": json.dumps(tool_result)
        }
    ]
    
logistic_response = openai.beta.chat.completions.parse(
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    messages=logistic_message,
    tools=tools,
    tool_choice="none",
    response_format=TravelRecommendations
)

typed_logistic_response: TravelRecommendations = logistic_response.choices[0].message.parsed

print("Typed logistic object:", typed_logistic_response)
print("Logistic PersonalisedNote", typed_logistic_response.personalized_note)

# Step 2: Logistics Planner builds the trip plan
# logistics_response = await logistics_agent.run(
#     f"Plan a week-long trip based on this recommendation:\n{dest_response}"
# )


Typed destination object: recommendations=[DestinationRecommendation(destination='Maldives', available=False, best_season='', highlights=[], estimated_budget_usd=0)] personalized_note="It looks like there was an issue retrieving information for the Maldives. However, for a budget of $5000, the Maldives is generally considered a dreamy destination known for its luxury resorts, white-sand beaches, and crystal-clear waters. Typically, this budget would cover a comfortable stay at a mid-range resort, including some activities like snorkeling or diving. It's best enjoyed during the dry season from November to April. You may want to consider reaching out to your travel agent for detailed availability and more personalized recommendations."
Destination PersonalisedNote It looks like there was an issue retrieving information for the Maldives. However, for a budget of $5000, the Maldives is generally considered a dreamy destination known for its luxury resorts, white-sand beaches, and crystal-c

## Summary

In this lesson we applied three agentic design patterns to a travel recommender scenario:

| Pattern | Key Idea | Benefit |
|---|---|---|
| **Clear Instructions** | Define persona, responsibilities, and constraints up front | Consistent, on-brand agent behavior |
| **Structured Output** | Use Pydantic models as the response format | Validated, machine-readable results |
| **Single Responsibility** | Give each agent one focused job | Easier to test, maintain, and compose |

These patterns compose naturally — you can combine clear instructions with structured output inside a single-responsibility agent to build robust, production-ready systems.